# VADER Sentiment Analysis — CSAT Data

VADER is a sentiment analysis tool that reads text and tells you if it's positive, negative, or neutral. It scores each response from -1 (very negative) to +1 (very positive). No training needed — it just works.

Same approach as NPS — checking if the text customers wrote matches what the EDA showed numerically. Difference here: no translated text column, so non-English responses get flagged as unscored.

In [ ]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from langdetect import detect, LangDetectException

analyzer = SentimentIntensityAnalyzer()

## Load the data

In [ ]:
df = pd.read_csv("/Users/Apple/Desktop/Hubspot/data/processed/csat.csv")
print(f"Loaded {len(df)} rows")
df.head(10)

## Pick which text to score
No translation column here, so we just check if the response text is English.
If it's not English or it's empty → unscored.

In [ ]:
def is_english(text):
    try:
        return detect(str(text)) == "en"
    except:
        return False

def get_text_to_score(text):
    if pd.notna(text) and str(text).strip():
        if is_english(text):
            return str(text).strip()
    return None

df["text_to_score"] = df["Response Text"].apply(get_text_to_score)

print(f"Scoreable: {df['text_to_score'].notna().sum()}")
print(f"Unscored: {df['text_to_score'].isna().sum()}")

## Run VADER
Each response gets a compound score. Then I label it:
- Above 0.05 → positive
- Below -0.05 → negative
- In between → neutral

Heads up — this is ~528k rows so it might take a few minutes.

In [ ]:
def analyze_sentiment(text):
    if text is None or pd.isna(text):
        return pd.Series({"vader_compound": None, "sentiment_label": "unscored"})

    scores = analyzer.polarity_scores(text)

    if scores["compound"] >= 0.05:
        label = "positive"
    elif scores["compound"] <= -0.05:
        label = "negative"
    else:
        label = "neutral"

    return pd.Series({"vader_compound": scores["compound"], "sentiment_label": label})

results = df["text_to_score"].apply(analyze_sentiment)
df = pd.concat([df, results], axis=1)

df[["Response Text", "vader_compound", "sentiment_label"]].head(10)

## Sentiment by Taxonomy Type

In [ ]:
scored_df = df[df["sentiment_label"] != "unscored"]

summary_tax = scored_df.groupby(["Taxonomy Type", "sentiment_label"]).size().unstack(fill_value=0)

for col in ["positive", "negative", "neutral"]:
    if col not in summary_tax.columns:
        summary_tax[col] = 0

summary_tax = summary_tax[["positive", "negative", "neutral"]]
summary_tax["total"] = summary_tax.sum(axis=1)
summary_tax["positive_pct"] = (summary_tax["positive"] / summary_tax["total"] * 100).round(1)
summary_tax["negative_pct"] = (summary_tax["negative"] / summary_tax["total"] * 100).round(1)
summary_tax["net_sentiment"] = (summary_tax["positive_pct"] - summary_tax["negative_pct"]).round(1)
summary_tax = summary_tax.sort_values("net_sentiment", ascending=False)

summary_tax

## Sentiment by Survey Name
This tells us which part of the product the feedback is about.

In [ ]:
summary_survey = scored_df.groupby(["Survey Name", "sentiment_label"]).size().unstack(fill_value=0)

for col in ["positive", "negative", "neutral"]:
    if col not in summary_survey.columns:
        summary_survey[col] = 0

summary_survey = summary_survey[["positive", "negative", "neutral"]]
summary_survey["total"] = summary_survey.sum(axis=1)
summary_survey["positive_pct"] = (summary_survey["positive"] / summary_survey["total"] * 100).round(1)
summary_survey["negative_pct"] = (summary_survey["negative"] / summary_survey["total"] * 100).round(1)
summary_survey["net_sentiment"] = (summary_survey["positive_pct"] - summary_survey["negative_pct"]).round(1)
summary_survey = summary_survey.sort_values("net_sentiment", ascending=False)

summary_survey

## Overall numbers

In [ ]:
total = len(scored_df)
print(f"Positive:  {(scored_df['sentiment_label'] == 'positive').sum()} ({(scored_df['sentiment_label'] == 'positive').sum()/total*100:.1f}%)")
print(f"Negative:  {(scored_df['sentiment_label'] == 'negative').sum()} ({(scored_df['sentiment_label'] == 'negative').sum()/total*100:.1f}%)")
print(f"Neutral:   {(scored_df['sentiment_label'] == 'neutral').sum()} ({(scored_df['sentiment_label'] == 'neutral').sum()/total*100:.1f}%)")
print(f"Unscored:  {df['text_to_score'].isna().sum()}")
print(f"Total:     {len(df)}")

## Cross-reference with EDA
The EDA showed Reliability (2.10 avg) and Expectation Gap (2.27 avg) as the weakest taxonomy areas.
VADER confirms this — Reliability and Expectation Gap have the worst net sentiment in the text too.

In [ ]:
df.drop(columns=["text_to_score"]).to_csv("csat_sentiment_results.csv", index=False)
summary_tax.to_csv("csat_sentiment_summary_taxonomy.csv")
summary_survey.to_csv("csat_sentiment_summary_survey.csv")
print("Saved: csat_sentiment_results.csv, csat_sentiment_summary_taxonomy.csv, csat_sentiment_summary_survey.csv")